# Week 1 · Day 1 — Web Summarizer with an LLM

## What we are building

A simple "Reader's Digest" web browser: give it a URL, it returns a concise markdown summary.

This single, small project touches every fundamental skill in LLM engineering:

| Skill | Where it appears |
|---|---|
| Web scraping | `scraper.py` — `fetch_website_contents()` |
| Prompt engineering | system prompt + user prompt composition |
| API calls | `openai.chat.completions.create()` |
| Provider flexibility | one helper works with OpenAI **and** Azure OpenAI |
| Experiment logging | `eval/log_utils.py` — `log_run()` |

### Prerequisites

Complete these before continuing:

- `uv sync` has been run (`.venv` exists)
- `.env` at the project root contains `OPENAI_API_KEY` (and optionally Azure keys)
- Run `uv run python src/setup/run_diagnostics.py` — all checks green

After today you will be able to call any Frontier LLM, craft prompts, and log your experiments.

## How to run this notebook

**In VS Code / Cursor:**
1. `View → Extensions` → install **Python** (ms-python) and **Jupyter** (ms-toolsai) if missing
2. Click **Select Kernel** (top-right) → **Python Environments…** → choose `.venv (Python 3.12.x)` — marked "Recommended"

**In JupyterLab:**
```bash
uv run jupyter lab
```
Then open `src/week1/day1/day1.ipynb`.

> **Tip:** Run cells top-to-bottom with **Shift + Enter**. A `NameError` usually means you skipped a cell above.

In [ ]:
import os
import sys
import time
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

# Repo root (pyproject.toml) + Day 1 folder (scraper + eval live here)
_here = Path().resolve()
_root = next((p for p in [_here, *_here.parents] if (p / "pyproject.toml").exists()), _here)
_day1 = _root / "src" / "week1" / "day1"
if str(_day1) not in sys.path:
    sys.path.insert(0, str(_day1))

from scraper import fetch_website_contents  # noqa: E402
from eval.log_utils import log_run  # noqa: E402

print(f"Repo root: {_root}")
print(f"Day 1 dir: {_day1}")
print("Imports OK")

## Provider Setup

We load credentials from `.env` and build an `OpenAI` client. The same pattern works for **Azure OpenAI** — swap `OpenAI()` for `AzureOpenAI(...)`.

```
OPENAI_API_KEY=sk-proj-…            # required for OpenAI path
AZURE_OPENAI_ENDPOINT=https://…     # required for Azure path
AZURE_OPENAI_API_KEY=…
AZURE_OPENAI_API_VERSION=2024-12-01-preview
AZURE_OPENAI_DEPLOYMENT_NAME=gpt-4o-mini
```

> **Cost note:** `gpt-4o-mini` is the cheapest capable model. All API calls in this notebook cost a few cents total.

**Troubleshooting:** If the next cell prints an error, open `src/setup/03_troubleshooting.ipynb` for step-by-step diagnosis.

In [ ]:
load_dotenv(override=True)

openai_key = os.getenv("OPENAI_API_KEY", "")
azure_key = os.getenv("AZURE_OPENAI_API_KEY", "")
azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT", "")

if openai_key:
    display_key = openai_key[:12] + "…"
    print(f"✅ OPENAI_API_KEY found: {display_key}")
else:
    print("⚠️  OPENAI_API_KEY not set — OpenAI path will fail")

if azure_key and azure_endpoint:
    print(f"✅ Azure OpenAI credentials found (endpoint: {azure_endpoint[:40]}…)")
elif azure_key or azure_endpoint:
    print("⚠️  Azure OpenAI partially configured — need both KEY and ENDPOINT")
else:
    print("ℹ️  Azure OpenAI not configured (optional)")

if not openai_key and not (azure_key and azure_endpoint):
    raise EnvironmentError(
        "No provider configured — set OPENAI_API_KEY or Azure credentials in .env"
    )

## Step 1 — Create a client and make your first API call

`OpenAI()` reads `OPENAI_API_KEY` from the environment automatically.  
Every call returns a `ChatCompletion` object — we extract the text via `.choices[0].message.content`.

In [ ]:
# Preview: this is all it takes to call an LLM.
# (The next cell creates the client and actually sends this to the API.)

message = "Hello! What is LLM Engineering in one sentence?"
messages = [{"role": "user", "content": message}]

messages  # show the structure

In [ ]:
client = OpenAI()  # reads OPENAI_API_KEY from environment

# Quick sanity check — cheapest capable model
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Say hello in one word."}],
    max_tokens=10,
)
print("Provider check:", response.choices[0].message.content)

## Step 2 — Scrape a page and build the summarizer pipeline

Now that the client works, we scrape a real page and pass its text to the LLM.

In [ ]:
# Let's try out this utility

ed = fetch_website_contents("https://edwarddonner.com")
print(ed)

## Types of prompts

You may know this already - but if not, you will get very familiar with it!

Models like GPT have been trained to receive instructions in a particular way.

They expect to receive:

**A system prompt** that tells them what task they are performing and what tone they should use

**A user prompt** -- the conversation starter that they should reply to

In [ ]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

In [ ]:
# Define our user prompt

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

## Messages

The API from OpenAI expects to receive messages in a particular structure.
Many of the other APIs share this structure:

```python
[
    {"role": "system", "content": "system message goes here"},
    {"role": "user", "content": "user message goes here"}
]
```
To give you a preview, the next 2 cells make a rather simple call - we won't stretch the mighty GPT (yet!)

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What is 2 + 2?"},
]

response = client.chat.completions.create(model="gpt-4o-mini", messages=messages)
response.choices[0].message.content

## Build the messages payload — and wire it into `summarize()`

The `messages_for()` helper composes exactly the structure the Chat Completions API expects: a `system` role that sets the persona, followed by a `user` role carrying the page content.

In [ ]:
# See how this function creates exactly the format above

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

In [ ]:
# Try this out, and then try for a few more websites

messages_for(ed)

## Step 3 — Wire scraper + prompts + API into one function

`summarize(url)` fetches the page, builds the messages, calls the model, logs the run, and returns markdown text.

In [ ]:
MODEL = "gpt-4o-mini"


def summarize(url: str) -> str:
    """Fetch a page, summarise it with an LLM, and log the experiment."""
    website = fetch_website_contents(url)
    t0 = time.monotonic()
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages_for(website),
    )
    latency_ms = (time.monotonic() - t0) * 1000
    summary = response.choices[0].message.content

    log_run(
        {
            "task": "web-summarize",
            "model": f"openai:{MODEL}",
            "prompt_hash": str(hash(url))[:8],
            "temperature": 1.0,
            "input_tokens": response.usage.prompt_tokens,
            "output_tokens": response.usage.completion_tokens,
            "latency_ms": round(latency_ms, 1),
            "score_relevance": "",
            "notes": url,
        }
    )
    return summary

In [ ]:
summarize("https://edwarddonner.com")

In [ ]:
# A function to display this nicely in the output, using markdown

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [ ]:
display_summary("https://edwarddonner.com")

## Try more websites

The plain `requests`+`BeautifulSoup` scraper works for most static sites.

**Limitations to be aware of:**

- Sites rendered with JavaScript (React/Next.js apps) return skeleton HTML — you'll get little text
- Sites behind Cloudflare or CDN protection may return `403 Forbidden`
- Login-gated content is not accessible

In [ ]:
display_summary("https://cnn.com")

In [ ]:
display_summary("https://anthropic.com")

## Business applications

> **Summarization** is one of the most widely deployed Gen AI use cases — it appears in email clients, research tools, news aggregators, due-diligence platforms, and customer support dashboards.

With just ~30 lines we have a working prototype. In production you would add:

- **Chunking** — pages longer than the context window need to be split and aggregated
- **Caching** — avoid re-summarising the same URL twice
- **Structured output** — ask the model to return JSON so you can index/filter summaries
- **Evaluation** — use `eval/metrics.py` to score relevance; track with `eval/log_utils.py`

### Exercise — try it yourself

Use the scaffolded cell below to build your own summarizer variant. Ideas:

- Email subject-line suggester (paste an email body → get a short subject)
- Competitor-watch tool (summarise 3 competitor homepages side-by-side)
- News digest bot (fetch an RSS/HTML news page → bullet list of top stories)

In [ ]:
# YOUR TURN — build your own summarizer variant
# Example: an email subject-line suggester

my_system_prompt = "You suggest a concise email subject line."
my_user_prompt = """
    Email body:
    Hi team, please review the attached Q3 results ...
"""

# Step 1: Build the messages list
my_messages = [
    {"role": "system", "content": my_system_prompt},
    {"role": "user", "content": my_user_prompt},
]

# Step 2: Call the API
# response = client.chat.completions.create(model=MODEL, messages=my_messages)

# Step 3: Print the result
# print(response.choices[0].message.content)

## Extension exercise — handle JavaScript-rendered pages

`https://openai.com` is a React app: our basic scraper returns mostly empty content.

**Approaches to explore:**
- **Selenium** or **Playwright** — spin up a real browser, wait for JS to execute, then scrape
- **httpx + playwright-async** — faster async variant
- **Jina Reader API** — `r.jina.ai/<url>` converts any page to clean markdown for free

Try upgrading `scraper.py` to handle at least one of these cases and test with `openai.com`.

## What's next?

You completed Day 1. Here's what we built and where to go next:

| Done today | Files |
|---|---|
| Web scraper | `src/week1/day1/scraper.py` |
| LLM summarizer | `src/week1/day1/day1.ipynb` |
| Experiment log | `eval/experiment_log.csv` — auto-updated on each `summarize()` call |

### Day 2 — deepen your prompt engineering skills

Open `source-material/week1/day2.ipynb` to study multi-turn conversations, system prompt tuning, and temperature experiments. Synthesise your own version in a new `src/week1/day1/day2_experiments.ipynb`.

### Check your experiment log

```python
import pandas as pd
pd.read_csv("eval/experiment_log.csv")
```

### Commit your work

```bash
git add src/week1/day1/ eval/experiment_log.csv
git commit -m "week1/day1: web summarizer with experiment logging"
```